In [1]:
import os, importlib.util
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
VOL_TEST = "/tmp/vol_test"

def find_file(name):
    """Cari file di /kaggle/input tanpa masuk ke folder kompetisi."""
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

def load_module(name, fname):
    path = find_file(fname)
    print(f"{fname:25s} -> {path}")
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

pp = load_module("pp", "03_preprocess_mri.py")    # preprocessing dari notebook 02
tb = load_module("tb", "04_train_baseline.py")    # definisi model dari notebook 03
WEIGHTS = find_file("efficientnet_b0_best.pt")
print("Bobot model               ->", WEIGHTS)
LABELS = tb.LABELS

03_preprocess_mri.py      -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/03_preprocess_mri.py
04_train_baseline.py      -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-03-train-baseline/04_train_baseline.py
Bobot model               -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-03-train-baseline/efficientnet_b0_best.pt


In [2]:
pp.SPLIT, pp.OUT_DIR = "test", VOL_TEST
os.makedirs(VOL_TEST, exist_ok=True)

series = pd.read_csv(f"{COMP}/test_series.csv")
failed = 0
for uid, g in tqdm(series.groupby("StudyInstanceUID"), desc="Preprocess test"):
    try:
        pp.process_study(uid, g)
    except Exception as e:
        failed += 1
        print("Gagal:", uid, type(e).__name__, e)

test = pd.read_csv(f"{COMP}/test.csv")
uids = test["StudyInstanceUID"].tolist()
ok = [u for u in uids if os.path.exists(f"{VOL_TEST}/{u}.npz")]
print(f"Test: {len(uids)} studi | volume tersedia: {len(ok)} | gagal: {failed}")
assert len(ok) >= 0.95 * len(uids), "Terlalu banyak studi gagal diproses, jangan disubmit"

Preprocess test:   0%|          | 0/3 [00:00<?, ?it/s]

Test: 3 studi | volume tersedia: 3 | gagal: 0


In [3]:
net = tb.KneeNet("efficientnet_b0", pretrained=False)          # struktur saja, tanpa download
net.load_state_dict(torch.load(WEIGHTS, map_location="cpu"))
net = net.cuda().to(memory_format=torch.channels_last).eval()

ds = tb.KneeDS(ok, np.full((len(ok), len(LABELS)), np.nan, np.float32), VOL_TEST, train=False)
dl = torch.utils.data.DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)
p = tb.predict(net, dl)
print("Bentuk prediksi:", p.shape)

Bentuk prediksi: (3, 12)


In [4]:
pred = pd.DataFrame(p, columns=LABELS)
pred.insert(0, "StudyInstanceUID", ok)

sub = test[["StudyInstanceUID"]].merge(pred, on="StudyInstanceUID", how="left")
n_missing = sub[LABELS].isna().any(axis=1).sum()
sub[LABELS] = sub[LABELS].fillna(0.5)            # jaga-jaga kalau ada studi gagal diproses

sample = pd.read_csv(f"{COMP}/sample_submission.csv")
sub = sub[sample.columns]
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("Studi tanpa prediksi (diisi 0.5):", n_missing)
print("Shape submission:", sub.shape, "| shape sample:", sample.shape)
print(sub.head())

Studi tanpa prediksi (diisi 0.5): 0
Shape submission: (3, 13) | shape sample: (3, 13)
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.002226  0.005620   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.710126  0.074500   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.000685  0.002314   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.157525          0.007696   0.223208    0.018021  0.500274   
1         0.569075          0.503113   0.018978    0.021615  0.038250   
2         0.670393          0.127218   0.236693    0.004382  0.523024   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.533094   0.007317  0.410874   0.004332  0.000517  
1  0.275519   0.526510  0.003839   0.133754  0.099295  
2  0.094845   0.003693  0.036631   0.006004  0.002758  
